# Voxa: train the Nigerian listening model on Google Colab

This notebook trains Voxa's on-device listening model for meetings where people mix languages:

- **North**: Hausa and English, plus Pidgin
- **South**: Pidgin and English

Run the cells **in order, from top to bottom**. Each step says what it does and roughly how long it takes.

**If Colab disconnects** (expect it on long runs): reconnect, then run every cell again from the top. Work that is already finished is skipped, and training carries on from the last finished epoch, because the model and its settings are saved in your Google Drive.

### Before you start

1. **Runtime → Change runtime type → T4 GPU** (or any better GPU).
2. A **Hugging Face account**. While signed in, open each page below and accept its terms (you only do this once):
   - AfriSpeech-200 (Nigerian-accented English): https://huggingface.co/datasets/intronhealth/afrispeech-200
   - NaijaVoices (Hausa, North only): https://huggingface.co/datasets/naijavoices/naijavoices-dataset
3. A **Hugging Face token**: huggingface.co → Settings → Access Tokens → New token (Read). In Colab, open the 🔑 **Secrets** panel on the left, add it as `HF_TOKEN`, and switch on notebook access.
4. About **10 GB free in Google Drive** for the model and checkpoints (about 40 GB more if you tick `KEEP_DATA_ON_DRIVE`).

**Licences:** NaijaVoices and AfriSpeech-200 are for non-commercial research. You can build and test a model trained on them, but you need a licence from their owners to ship it in a product you sell.

## 1. Settings

Change these if you need to, then run the cell. Run it again after every reconnect.

- **REGION**: `north` (Hausa + English) or `south` (Pidgin + English).
- **MODEL_SIZE**: `small` runs on 2 GB phones; `medium` is more accurate and needs a better phone and longer training.
- **Hours**: how much audio to download. More hours give a better model and take longer to train. Start small (the defaults) and add more once everything works.
- **MAX_DURATION**: seconds of audio per training batch. 300 fits a T4; use 600 on an A100.
- **KEEP_DATA_ON_DRIVE**: saves the prepared audio to Drive too (about 40 GB). After a reconnect, training then restarts in minutes instead of spending an hour downloading and preparing again.
- **BRANCH**: the branch of the Voxa repository that holds the training code you want to run.

In [ ]:
#@title Settings
REGION = "north"  #@param ["north", "south"]
MODEL_SIZE = "small"  #@param ["small", "medium"]
#@markdown Hours of training audio to download
LOCAL_HOURS = 60  #@param {type:"number"}
ENGLISH_HOURS = 40  #@param {type:"number"}
SYNTH_HOURS = 30  #@param {type:"number"}
#@markdown Passes over the data: stage A is English and Pidgin, stage B is the region model
STAGE_A_EPOCHS = 10  #@param {type:"integer"}
STAGE_B_EPOCHS = 20  #@param {type:"integer"}
MAX_DURATION = 300  #@param {type:"integer"}
KEEP_DATA_ON_DRIVE = False  #@param {type:"boolean"}
REPO = "https://github.com/ysddesigns/voxa.git"  #@param {type:"string"}
BRANCH = "feat/listen-accuracy"  #@param {type:"string"}
DRIVE_FOLDER = "voxa-training"  #@param {type:"string"}

import os

CODE = "/content/voxa"
ASR = f"{CODE}/training/asr"
DATA = "/content/data"
DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
SAVE = f"{DRIVE}/{REGION}-{MODEL_SIZE}"   # model, checkpoints, vocabulary, scores
MINE = f"{DRIVE}/my-data"                # your own recordings (step 8)
CACHE = f"{DRIVE}/data-cache-{REGION}"   # prepared data, if KEEP_DATA_ON_DRIVE

os.environ.update({
    "REGION": REGION, "MODEL_SIZE": MODEL_SIZE, "CODE": CODE, "ASR": ASR, "DATA": DATA,
    "SAVE": SAVE, "MINE": MINE, "CACHE": CACHE, "REPO": REPO, "BRANCH": BRANCH,
    "LOCAL_HOURS": str(LOCAL_HOURS), "ENGLISH_HOURS": str(ENGLISH_HOURS), "SYNTH_HOURS": str(SYNTH_HOURS),
    "STAGE_A_EPOCHS": str(STAGE_A_EPOCHS), "STAGE_B_EPOCHS": str(STAGE_B_EPOCHS), "SOUTH_EPOCHS": str(STAGE_B_EPOCHS),
    "MAX_DURATION": str(MAX_DURATION), "KEEP_DATA_ON_DRIVE": "1" if KEEP_DATA_ON_DRIVE else "0",
    "DATA_DIR": DATA, "VOCAB_SIZE": "2000",
    # Checkpoints are large: keep the last 4 epochs (the export averages 3) and one mid-epoch save.
    "KEEP_EPOCHS": "4", "KEEP_LAST_K": "1",
})
print(f"Training the {REGION} model ({MODEL_SIZE}). Saved to Google Drive: {SAVE}")

## 2. Check the GPU

This should print a GPU name such as Tesla T4. If it fails, choose **Runtime → Change runtime type → T4 GPU**, then run from the top.

In [ ]:
import subprocess

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True)
assert r.returncode == 0, "No GPU. Choose Runtime > Change runtime type > T4 GPU, then run the cells from the top."
print("GPU:", r.stdout.strip())

## 3. Connect Google Drive

Allow access when asked. The trained model, checkpoints and scores are kept in Drive, so nothing is lost when Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
for d in (SAVE, MINE):
    os.makedirs(d, exist_ok=True)
print("Saving to:", SAVE)
print("Your own recordings go in:", MINE, "(see step 8)")

## 4. Get the Voxa code

This downloads the training scripts from the branch set in step 1. On a re-run, it fetches the latest version of that branch.

In [ ]:
%%bash
set -e
if [ -d "$CODE/.git" ]; then
  git -C "$CODE" fetch -q origin "$BRANCH"
  git -C "$CODE" checkout -q "$BRANCH"
  git -C "$CODE" reset -q --hard "origin/$BRANCH"
else
  git clone -q --branch "$BRANCH" "$REPO" "$CODE"
fi
git -C "$CODE" log --oneline -1
if [ ! -f "$ASR/scripts/fetch_data.py" ]; then
  echo "This branch does not have the latest training code (scripts/fetch_data.py is missing)."
  echo "Push the training changes to $BRANCH, or set BRANCH in step 1 to the branch that has them."
  exit 1
fi

## 5. Install the training tools (about 10 minutes)

This installs k2 to match the PyTorch on this machine, then icefall, lhotse and sherpa-onnx, and downloads the starting models. It also links the `exp`, `splits`, `packs` and `reports` folders to Google Drive, so checkpoints and scores survive a disconnect.

In [ ]:
%%bash
set -e
cd "$ASR"
python scripts/colab_install.py
bash setup.sh > /content/setup.log 2>&1 || { tail -40 /content/setup.log; exit 1; }
tail -3 /content/setup.log

# Keep what must survive a disconnect in Drive.
for d in exp splits packs reports; do
  mkdir -p "$SAVE/$d"
  if [ -d "$d" ] && [ ! -L "$d" ]; then cp -rn "$d/." "$SAVE/$d/" 2>/dev/null || true; rm -rf "$d"; fi
  ln -sfn "$SAVE/$d" "$d"
done
# The word-piece vocabulary too: a checkpoint only fits the vocabulary it was trained with.
mkdir -p data
for v in "lang_bpe_north_$VOCAB_SIZE" "lang_bpe_$VOCAB_SIZE"; do
  mkdir -p "$SAVE/lang/$v"
  [ -L "data/$v" ] || rm -rf "data/$v"
  ln -sfn "$SAVE/lang/$v" "data/$v"
done
echo "Ready."

## 6. Sign in to Hugging Face

This reads the `HF_TOKEN` secret (or asks for the token), then checks that you have access to each dataset this region needs. If a dataset shows **NO ACCESS**, open its page, accept the terms, and run this cell again.

In [ ]:
from getpass import getpass
from huggingface_hub import hf_hub_download

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
token = token or getpass("Hugging Face token (Read): ")
os.environ["HF_TOKEN"] = token

needed = ["intronhealth/afrispeech-200", "michaelodafe/pidgin-asr-combined"]
if REGION == "north":
    needed += ["naijavoices/naijavoices-dataset", "google/fleurs"]
missing = []
for repo in needed:
    try:
        hf_hub_download(repo, "README.md", repo_type="dataset", token=token, local_dir="/content/hf-check")
        print("OK         ", repo)
    except Exception as e:
        missing.append(repo)
        print("NO ACCESS  ", repo, "->", f"https://huggingface.co/datasets/{repo}", f"({type(e).__name__})")
assert not missing, "Accept the terms on the pages above (signed in to Hugging Face), then run this cell again."

## 7. Download the training data (30 to 90 minutes)

Each dataset is downloaded one file at a time and turned into 16 kHz audio. Run this again after a disconnect and it carries on where it stopped.

| Region | Data |
|---|---|
| North | Hausa (NaijaVoices, FLEURS), English read by Hausa and other northern speakers (AfriSpeech-200), Pidgin |
| South | English read by Yoruba, Igbo, Edo, Ijaw and other southern speakers (AfriSpeech-200), Pidgin |

In [ ]:
%%bash
set -e
cd "$ASR"
mkdir -p "$DATA/fetched"
if [ "$KEEP_DATA_ON_DRIVE" = 1 ] && [ -d "$CACHE" ]; then
  echo "Restoring prepared data from Drive..."
  mkdir -p data
  [ -d "$CACHE/data" ] && cp -ru "$CACHE/data/." "$DATA/"
  for d in manifests fbank codeswitch; do [ -d "$CACHE/$d" ] && mkdir -p "data/$d" && cp -ru "$CACHE/$d/." "data/$d/"; done
fi

python scripts/fetch_data.py pidgin --out "$DATA/fetched"
if [ "$REGION" = north ]; then
  python scripts/fetch_data.py afrispeech --accents north --hours "$ENGLISH_HOURS" --out "$DATA/fetched"
  python scripts/fetch_data.py naijavoices --language ha --hours "$LOCAL_HOURS" --out "$DATA/fetched"
  python scripts/fetch_data.py fleurs --code ha_ng --out "$DATA/fleurs"
else
  python scripts/fetch_data.py afrispeech --accents south --hours "$ENGLISH_HOURS" --out "$DATA/fetched"
fi
df -h /content | tail -1

## 8. Add your own recordings (optional, but needed to measure mixed speech)

Downloaded datasets have **no mixed-language speech**. Only your own recordings can test whether the model follows a switch from Hausa to English, and they also teach it. Put them in the Drive folder printed in step 3 (`my-data`):

- **Donate your voice zips** (`voxa-voice-....zip`): drop them in as they are.
- **Transcribed meetings**: a `cs-<name>.jsonl` file (see "Transcribing code-switched meetings" in `training/asr/README.md`) and the audio folder it points to.
- **`test-speakers.txt`**: one speaker id per line. These speakers are always kept for testing and never trained on. List the speakers of your meeting recordings here.

Do this **before the first run of step 9**. Test speakers are chosen once and then locked, so speakers added later go to training only.

In [ ]:
%%bash
set -e
mkdir -p "$DATA/donated"
shopt -s nullglob
n=0
for z in "$MINE"/*.zip; do
  name="$(basename "$z" .zip)"
  [ -d "$DATA/donated/$name" ] || unzip -q -o "$z" -d "$DATA/donated/$name"
  n=$((n+1))
done
for f in "$MINE"/*; do
  case "$f" in *.zip) continue ;; esac
  cp -ru "$f" "$DATA/"
done
echo "Donated zips: $n"
ls "$DATA" | sed 's/^/  /'
[ -f "$DATA/test-speakers.txt" ] && echo "Test speakers: $(grep -c . "$DATA/test-speakers.txt")" || echo "No test-speakers.txt: mixed-language scores will be missing."

## 9. Prepare the data (30 to 90 minutes the first time)

This locks the test speakers, builds the training lists, joins Hausa and English clips into mixed ones, builds the word pieces, and computes audio features. It is skipped on a re-run when nothing has changed.

In [ ]:
%%bash
set -e
cd "$ASR"
set -o pipefail
STOP_AFTER_PREP=1 bash scripts/train_cloud.sh 2>&1 | tee -a "$SAVE/prepare.log"
if [ "$KEEP_DATA_ON_DRIVE" = 1 ]; then
  echo "Saving prepared data to Drive (first time only takes a while)..."
  mkdir -p "$CACHE/data"
  cp -ru "$DATA/." "$CACHE/data/"
  for d in manifests fbank codeswitch; do [ -d "data/$d" ] && mkdir -p "$CACHE/$d" && cp -ru "data/$d/." "$CACHE/$d/"; done
fi
echo "Data ready."

## 10. Train (several hours; safe to run again after a disconnect)

Training runs in two stages. Stage A learns Nigerian English and Pidgin. Stage B is the region model, which also learns Hausa (North) and mixed speech. Each epoch is saved to Drive as it finishes. If Colab disconnects, run every cell again from the top, and this one carries on from the last saved epoch.

The log shows a `loss` that should go down. To see how long one epoch takes, compare the times of two `Epoch` lines. Keep this browser tab open: free Colab stops idle sessions.

Optional: run the next cell first to watch training in TensorBoard.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {ASR}/exp

In [ ]:
!cd {ASR} && bash scripts/train_cloud.sh 2>&1 | tee -a {SAVE}/train.log

## 11. Results

Word error per language on the locked test speakers (lower is better). For mixed-speech sets (`cs-ha`, `cs-pcm`), it also shows the errors on English words and on local-language words inside mixed sentences. `-full` is the after-meeting model.

In [ ]:
import glob
import json

for path in sorted(glob.glob(f"{ASR}/reports/voxa-{REGION}-*.json")):
    r = json.load(open(path, encoding="utf-8"))
    print(f"\n{os.path.basename(path)}  ({r.get('decoding')}, {r.get('split')} set)")
    for lang, s in r["languages"].items():
        print(f"  {lang:7} word error {s['wer']:.1%}  over {s['words']} words, {s['hours']} h")
        for k, v in (s.get("by_word_language") or {}).items():
            if v["words"] and k not in ("shared", "unknown"):
                print(f"           {k} words inside mixed sentences: {v['error']:.1%} wrong of {v['words']}")

## 12. Download the model

The packs are already in your Drive (`packs` in the folder from step 1). This cell also downloads them as one zip.

- `listen-ng-<region>`: live captions
- `listen-ng-<region>-full`: the more accurate after-meeting pass

To try one on a phone, upload the pack folder to a Hugging Face model repository. Then add the catalogue entry that step 7 of the log printed to `src/features/packs/catalogue.ts` as a lab pack, and compare it in the Engine lab on a 2 GB phone.

In [ ]:
!cd {ASR}/packs && zip -qr /content/voxa-{REGION}-packs.zip listen-ng-{REGION} listen-ng-{REGION}-full && ls -lh /content/voxa-{REGION}-packs.zip
from google.colab import files
files.download(f"/content/voxa-{REGION}-packs.zip")

## Troubleshooting

- **"CUDA out of memory"**: lower `MAX_DURATION` in step 1 (try 200), then run from the top. Training resumes.
- **A dataset fails with "No ... column"**: the dataset changed its layout. The message lists the columns it found. Add the new name to `TEXT_COLUMNS`, `SPEAKER_COLUMNS` or `PATH_COLUMNS` in `training/asr/voxa_asr/fetch.py`.
- **The disk is full** (`df -h` in step 7): lower the hours in step 1. Expect about 115 MB per hour of audio, plus features.
- **You want to start a stage over**: delete its folder in Drive (`exp/english-small` or `exp/north-small`).
- **You changed the hours after training started**: the new data is used from the next epoch on. The test speakers stay the same.